# MEIKA Vox · transcripción automática por carpeta
**Activa GPU → Ejecutar todo → autoriza Drive → elige una carpeta → Transcribir carpeta.**
No escribas nombres de archivos. El programa encuentra los audios en esa carpeta y sus subcarpetas.
Los resultados quedan en `Mi unidad/MEIKA_Vox/Transcripciones`. Si Colab se desconecta,
vuelve a ejecutar: omite los audios que ya terminaron con la misma configuración y reintenta los pendientes.

La primera instalación y descarga de modelos requiere internet y puede tardar varios minutos.
Colab debe permanecer conectado mientras procesa. Este notebook no se ejecuta continuamente en segundo plano.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/meika-vox")
RUNTIME = Path("/content/meika-vox-runtime")


def checked(cmd, **kwargs):
    subprocess.run([str(item) for item in cmd], check=True, **kwargs)


checked(["apt-get", "-qq", "update"])
checked(["apt-get", "-qq", "install", "-y", "ffmpeg", "git", "python3-venv"])
if not (REPO / ".git").exists():
    checked(["git", "clone", "https://github.com/MeikaLab/meika-vox.git", REPO])
else:
    checked(["git", "-C", REPO, "pull", "--ff-only"])
if not (RUNTIME / "bin/python").exists():
    checked([sys.executable, "-m", "venv", RUNTIME])
PYTHON = RUNTIME / "bin/python"
VOX = RUNTIME / "bin/meika-vox"
checked([PYTHON, "-m", "pip", "install", "--upgrade", "pip"])
checked([PYTHON, "-m", "pip", "install", "-e", str(REPO) + "[whisperx]"])
checked([PYTHON, "-c", "import nltk; nltk.download('punkt_tab', raise_on_error=True)"])
checked([VOX, "doctor", "--strict"])
print("Instalación comprobada. Ahora conecta Google Drive.")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import importlib.util
from collections import Counter

import ipywidgets as widgets
from IPython.display import display

ROOT = Path("/content/drive/MyDrive")
OUTPUT = ROOT / "MEIKA_Vox/Transcripciones"
spec = importlib.util.spec_from_file_location("vox_batch", REPO / "scripts/colab_batch.py")
batch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(batch)
print("Buscando carpetas con audios en Mi unidad…")
audios = batch.discover_audio(ROOT, exclude=OUTPUT)
counts = Counter(p.parent for p in audios)
if not counts:
    raise RuntimeError(
        "No encontré audios en Mi unidad. Guarda allí la carpeta y vuelve a ejecutar."
    )
choices = [
    (f"{p.relative_to(ROOT)} · {count} audios directos", str(p))
    for p, count in sorted(counts.items(), key=lambda item: str(item[0]))
]
folder_picker = widgets.Dropdown(
    options=choices, description="Carpeta:", layout=widgets.Layout(width="95%")
)
diarize_picker = widgets.Checkbox(value=False, description="Separar hablantes (requiere HF_TOKEN)")
force_picker = widgets.Checkbox(value=False, description="Volver a transcribir los ya terminados")
start_button = widgets.Button(
    description="Transcribir carpeta", button_style="success", layout=widgets.Layout(width="220px")
)
status = widgets.HTML()
log = widgets.Output()


def update_count(change=None):
    selected = batch.discover_audio(Path(folder_picker.value), exclude=OUTPUT)
    status.value = f"<b>{len(selected)} audios</b> en la carpeta elegida y sus subcarpetas."


folder_picker.observe(update_count, names="value")
update_count()


def start_batch(button):
    start_button.disabled = True
    folder_picker.disabled = True
    diarize_picker.disabled = True
    force_picker.disabled = True
    log.clear_output()
    with log:
        try:
            import json

            if diarize_picker.value:
                from getpass import getpass

                try:
                    from google.colab import userdata

                    token = userdata.get("HF_TOKEN")
                except Exception:
                    token = None
                token = token or getpass("HF_TOKEN (oculto): ").strip()
                if not token:
                    raise RuntimeError(
                        "Falta HF_TOKEN; desmarca separar hablantes para obtener texto."
                    )
                os.environ["HF_TOKEN"] = token
                print("Se requiere aceptar las condiciones del modelo pyannote community-1.")
            runtime = json.loads(
                subprocess.check_output([str(VOX), "doctor", "--strict"], text=True)
            )
            device = "cuda" if runtime["cuda_available"] else "cpu"
            model = "large-v3" if device == "cuda" else "small"
            glossary = REPO / "configs/glossaries/santa_maria_2026.json"
            command = [
                str(VOX),
                "--project-id",
                "SM26",
                "--language",
                "es",
                "--model",
                model,
                "--device",
                device,
                "--batch-size",
                "4" if device == "cuda" else "1",
                "--glossary",
                str(glossary),
            ]
            if diarize_picker.value:
                command.append("--diarize")
            revision = subprocess.check_output(
                ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
            )
            configuration = revision.strip() + batch.fingerprint(glossary)
            print(f"Motor: {model} · {device}. Mantén Colab conectado mientras trabaja.")
            report = batch.process_folder(
                Path(folder_picker.value),
                OUTPUT,
                command,
                force=force_picker.value,
                configuration=configuration,
            )
            if report["failed"]:
                print("Hay pendientes. Pulsa el botón otra vez para reintentarlos.")
            else:
                print("Listo. Abre Mi unidad → MEIKA_Vox → Transcripciones.")
        except Exception as exc:
            print("No se completó:", exc)
        finally:
            start_button.disabled = False
            folder_picker.disabled = False
            diarize_picker.disabled = False
            force_picker.disabled = False


start_button.on_click(start_batch)
display(widgets.VBox([folder_picker, status, diarize_picker, force_picker, start_button, log]))